# Week 68, Day 3: Cython

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w068d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ctypes

python_total = 0
c_total = ctypes.c_int32(0)
for _ in range(3):
    python_total += 1_000_000_000
    c_total.value += 1_000_000_000

print(python_total)
print(c_total.value)
print(-7 // 2, int(-7 / 2))

In [ ]:
import time

def count_primes(limit):
    count = 0
    for n in range(2, limit):
        d, is_prime = 2, True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 1
        count += is_prime
    return count

start = time.perf_counter()
print(count_primes(5000))
print("pure Python took", "under" if time.perf_counter() - start < 5 else "over", "5 seconds here; Cython with types would take a few milliseconds")

In [ ]:
TYPE_MAP = {"int": "long", "float": "double", "bool": "bint", "str": "str", "bytes": "bytes"}

def cython_signature(name, params, returns, kind="cpdef"):
    "Translate Python annotations into a Cython signature."
    args = ", ".join(f"{TYPE_MAP.get(t, 'object')} {p}" for p, t in params)
    return f"{kind} {TYPE_MAP.get(returns, 'object')} {name}({args}):"

print(cython_signature("area", [("w", "float"), ("h", "float")], "float"))
print(cython_signature("count", [("items", "list"), ("limit", "int")], "int", kind="cdef"))

In [ ]:
import ast

DYNAMIC_CALLS = {"eval", "exec", "getattr", "setattr", "globals", "locals"}

def slow_spots(source):
    found = []
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in DYNAMIC_CALLS:
            found.append((node.lineno, f"{node.func.id}()"))
        elif isinstance(node, (ast.Yield, ast.YieldFrom)):
            found.append((node.lineno, "generator"))
        elif isinstance(node, ast.Lambda):
            found.append((node.lineno, "lambda"))
    return sorted(found)

code = "def f(obj, name):\n    key = getattr(obj, name)\n    return sorted(obj, key=lambda x: x[key])\n"
print(slow_spots(code))

In [ ]:
from array import array

readings = array("d", [1.5, 2.5, 4.0])
view = memoryview(readings)
print(view.format, view.itemsize, view.nbytes, view.tolist())
view[1] = 9.0                         # writes into the array: no copy
print(readings.tolist())

raw = bytearray(8)
as_ints = memoryview(raw).cast("i")   # the same 8 bytes, seen as two C ints
as_ints[0] = 258
print(as_ints.tolist(), list(raw[:4]))

part = view[1:]                        # a slice is another view, not a copy
part[0] = 0.0
print(readings.tolist(), part.obj is readings)

In [ ]:
def c_div(a, b):
    "C's integer division: truncate towards zero."
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b >= 0) else -q

def c_mod(a, b):
    return a - b * c_div(a, b)

for a, b in [(7, 2), (-7, 2), (7, -2)]:
    print(f"{a:3} / {b:2}: Python {a // b:2} remainder {a % b:2}   C {c_div(a, b):2} remainder {c_mod(a, b):2}")

> **Key idea.** Cython compiles a Python superset (.pyx, or plain Python in pure-Python mode) to C that uses the C API correctly, then to an extension module; untyped code runs modestly faster, typed hot loops run at C speed. def functions are Python-callable, cdef functions are C-only and fast, cpdef gives both; cdef declares C variables (int, double, bint, Py_ssize_t) and typed objects. Untyped variables, Python calls, Python containers and dynamic features stay slow; cython -a shades lines by Python interaction. The buffer protocol shares raw memory without copying (memoryview in Python, typed memoryviews like double[:] in Cython). Typed code follows C rules: integers overflow, cdivision truncates and skips zero checks, boundscheck and wraparound can be turned off; with nogil releases the GIL and prange runs loops in parallel.

### Exercise: From annotations to Cython

Write `to_cython(source)`: for every function in some Python source whose parameters and return are annotated, produce its Cython signature line, mapping annotations with `C_TYPES` (`int` to `long`, `float` to `double`, `bool` to `bint`, anything else, or no annotation, to `object`): functions whose names start with `_` become `cdef`, the others `cpdef`. Return the list of lines like `cpdef double area(double w, double h):`, in source order. Then `needs_wrapper(line)`: True for `cpdef` and `def` lines (Python can call them), False for `cdef`.

In [ ]:
import ast

C_TYPES = {"int": "long", "float": "double", "bool": "bint"}

def to_cython(source):
    ___

def needs_wrapper(line):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_signature():\n    \"Cython signatures\"\n    src = (\n        \"def area(w: float, h: float) -> float:\\n    return w * h\\n\"\n        \"def _square(x: int) -> int:\\n    return x * x\\n\"\n        \"def label(name: str, n) -> str:\\n    return name * n\\n\"\n        \"def flag(ok: bool) -> bool:\\n    return not ok\\n\"\n    )\n    assert to_cython(src) == [\n        \"cpdef double area(double w, double h):\",\n        \"cdef long _square(long x):\",\n        \"cpdef object label(object name, object n):\",\n        \"cpdef bint flag(bint ok):\",\n    ]\n    assert to_cython(\"x = 1\") == []\n    assert needs_wrapper(\"cpdef double area(double w):\") and not needs_wrapper(\"cdef long f(long x):\")")

### Exercise: Finding what stays slow

Write `slow_features(source)`: a sorted list of `(line, feature)` pairs for the Python features inside functions that keep Cython from compiling code to plain C: `"*args"` and `"**kwargs"` parameters (reported on the `def` line), `"generator"` (any `yield`), `"lambda"`, `"closure"` (a function defined inside another function), and `"dynamic call"` for calls to `eval`, `exec`, `getattr` or `setattr`. Then `is_simple(source)`: True if there are none.

In [ ]:
import ast

def slow_features(source):
    ___

def is_simple(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_slowspots():\n    \"dynamic features\"\n    src = (\n        \"def total(*values, **options):\\n\"\n        \"    key = getattr(options, 'key', None)\\n\"\n        \"    def inner(x):\\n\"\n        \"        return x * 2\\n\"\n        \"    return sorted(values, key=lambda v: v)\\n\"\n        \"def stream(n):\\n\"\n        \"    for i in range(n):\\n\"\n        \"        yield i\\n\"\n        \"def plain(a, b):\\n\"\n        \"    return a * b + 1\\n\"\n    )\n    assert slow_features(src) == [(1, \"**kwargs\"), (1, \"*args\"), (2, \"dynamic call\"), (3, \"closure\"), (5, \"lambda\"), (8, \"generator\")]\n    assert is_simple(\"def plain(a, b):\\n    return a * b + 1\\n\") and not is_simple(src)")

### Exercise: The buffer protocol

Using `memoryview` (no copying of the data), write `total(buffer)`: the sum of a buffer of numbers (an `array.array` or anything supporting the buffer protocol with a numeric format); `scale_in_place(buffer, factor)`: multiply every element through a memoryview, changing the original; `as_format(raw, fmt)`: the elements of a `bytes` or `bytearray` reinterpreted with `memoryview(raw).cast(fmt)`, as a list; and `describe(buffer)`: a dict with the view's `format`, `itemsize`, `count` (number of elements) and `readonly`.

In [ ]:
def total(buffer):
    ___

def scale_in_place(buffer, factor):
    ___

def as_format(raw, fmt):
    ___

def describe(buffer):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_views():\n    \"memoryview\"\n    from array import array\n    a = array(\"d\", [1.5, 2.5, 4.0])\n    assert total(a) == 8.0 and total(array(\"i\", [1, 2, 3])) == 6\n    scale_in_place(a, 2)\n    assert a.tolist() == [3.0, 5.0, 8.0]\n    ints = array(\"i\", [1, 2])\n    scale_in_place(ints, 3)\n    assert ints.tolist() == [3, 6]\n    import struct\n    raw = struct.pack(\"=ii\", 258, 7)\n    assert as_format(raw, \"i\") == [258, 7] and as_format(bytes([1, 2, 3]), \"B\") == [1, 2, 3]\n    assert describe(a) == {\"format\": \"d\", \"itemsize\": 8, \"count\": 3, \"readonly\": False}\n    assert describe(b\"abc\")[\"readonly\"] is True and describe(b\"abc\")[\"count\"] == 3\n    try:\n        scale_in_place(bytearray(b\"ab\"), 300)\n        assert False\n    except (ValueError, TypeError):\n        pass")

### Exercise: C's integer rules

Model the arithmetic of typed Cython code. Write `c_add(a, b, bits=32)`: the sum wrapped to a signed integer of `bits` bits, as C would compute it; `c_div(a, b)` and `c_mod(a, b)`: C's division (truncating towards zero) and remainder (with the sign of `a`, so that `c_div(a, b) * b + c_mod(a, b) == a`), raising `ZeroDivisionError` for `b == 0`; and `differs_from_python(a, b)`: True if C's division or remainder gives a different answer from Python's `//` or `%`.

In [ ]:
def c_add(a, b, bits=32):
    ___

def c_div(a, b):
    ___

def c_mod(a, b):
    ___

def differs_from_python(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cmath():\n    \"C semantics\"\n    assert c_add(2 ** 31 - 1, 1) == -(2 ** 31) and c_add(100, 28, bits=8) == -128 and c_add(5, 6) == 11\n    assert c_add(-(2 ** 63), -1, bits=64) == 2 ** 63 - 1\n    assert c_div(7, 2) == 3 and c_div(-7, 2) == -3 and c_div(7, -2) == -3 and c_div(-7, -2) == 3\n    assert c_mod(-7, 2) == -1 and c_mod(7, -2) == 1 and c_mod(7, 2) == 1\n    for a, b in [(-7, 2), (7, -2), (9, 4), (-9, -4)]:\n        assert c_div(a, b) * b + c_mod(a, b) == a\n    assert differs_from_python(-7, 2) and not differs_from_python(7, 2) and not differs_from_python(-8, 2)\n    try:\n        c_div(1, 0)\n        assert False\n    except ZeroDivisionError:\n        pass")

### Exercise: Typing the hot loop

A profile shows which lines are hot. Given a list of `(line_text, runs, python_ops)` entries for a function (how many times each line runs, and how many Python object operations it does per run when untyped), write `python_cost(lines)`: the total `runs * python_ops`; `worth_typing(lines, share=0.8)`: the smallest set of line texts, taken from the costliest first (ties in their order in the list), whose combined cost reaches at least `share` of the total, as a list in that order; and `after_typing(lines, typed)`: the remaining cost if the lines in `typed` become plain C (cost 0).

In [ ]:
def python_cost(lines):
    ___

def worth_typing(lines, share=0.8):
    ___

def after_typing(lines, typed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hotloop():\n    \"where to add types\"\n    lines = [(\"count = 0\", 1, 2), (\"for n in range(limit):\", 10000, 3), (\"d = 2\", 10000, 1),\n             (\"while d * d <= n:\", 200000, 4), (\"if n % d == 0:\", 190000, 3), (\"d += 1\", 190000, 2),\n             (\"return count\", 1, 1)]\n    assert python_cost(lines) == 2 + 30000 + 10000 + 800000 + 570000 + 380000 + 1\n    assert worth_typing(lines) == [\"while d * d <= n:\", \"if n % d == 0:\", \"d += 1\"]\n    assert worth_typing(lines, share=0.4) == [\"while d * d <= n:\"]\n    assert after_typing(lines, worth_typing(lines)) == 40003\n    assert worth_typing([]) == []")

### Bug hunt: A C int that overflowed

The total is kept in a 32-bit C integer, as an untyped-to-`int` Cython variable would be, and adding three billions overflows it: this prints a negative number. Keep it in a 64-bit integer (`c_int64`, like Cython's `long long`) so it prints `3000000000`.

In [ ]:
import ctypes

total = ctypes.c_int32(0)
for amount in (1_000_000_000, 1_000_000_000, 1_000_000_000):
    total.value += amount
print(total.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a wide enough type\"\n    expect_output(\"3000000000\")")

### Bug hunt: A read-only buffer

`bytes` objects are immutable, so a memoryview of one is **read-only** and writing through it raises `TypeError`. Use a `bytearray`, whose buffer can be written, so this prints `b'Hello'`.

In [ ]:
data = b"hello"
view = memoryview(data)
view[0] = ord("H")
print(bytes(data))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a writable buffer\"\n    expect_output(\"b'Hello'\")")

### Bug hunt: Rounding the wrong way

This helper was meant to match Python's `//` (which rounds down), but `int(a / b)` truncates towards zero, as C does, so for negative numbers it's out by one: this prints `-3`. Use floor division so it prints `-4`.

In [ ]:
def floor_div(a, b):
    return int(a / b)

print(floor_div(-7, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"floor division\"\n    expect_output(\"-4\")")

### Puzzle: A typed Cython function

Arrange the lines of a Cython function (as you'd write it in a `.pyx` file on your own computer) that sums a typed memoryview of doubles. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the kind of Cython function callable only from Cython: "def", "cdef" or "cpdef"
q = ___   # the Cython type for a C integer used as a bool
r = ___   # C's -7 / 2 for integers, truncating towards zero
s = ___   # memoryview(b"abc").readonly
t = ___   # the command-line flag that makes Cython write its annotation report: "-a" or "-x"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"cdef\" and q == \"bint\" and r == -3, \"p, q and r\"\n    assert s is True and t == \"-a\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny translator to Cython

Write `cythonize_function(source)`: translate one annotated Python function into Cython pure-`.pyx` text:

- the signature line as in the first exercise (`cpdef` or `cdef` by name, mapping `int`, `float` and `bool`; others `object`);
- every **annotated local assignment** (`total: float = 0` or `i: int`) at the top level of the body becomes a `cdef` declaration line at the start of the body, `cdef double total = 0` or `cdef long i`, in order, using `ast.unparse` for the value; the rest of the body is kept, without those annotated lines (but an annotated assignment with a value becomes a `cdef` with that value, so it isn't repeated);
- the body is indented four spaces and produced with `ast.unparse` statement by statement (each statement's lines indented);
- raise `ValueError("one function expected")` unless the source contains exactly one function.

Return the text as one string, lines joined by newlines.

In [ ]:
import ast

C_TYPES = {"int": "long", "float": "double", "bool": "bint"}

def cythonize_function(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_translate():\n    \"Python to Cython\"\n    src = (\n        \"def total(n: int) -> float:\\n\"\n        \"    acc: float = 0\\n\"\n        \"    i: int\\n\"\n        \"    for i in range(n):\\n\"\n        \"        acc += i * 0.5\\n\"\n        \"    return acc\\n\"\n    )\n    assert cythonize_function(src) == (\n        \"cpdef double total(long n):\\n\"\n        \"    cdef double acc = 0\\n\"\n        \"    cdef long i\\n\"\n        \"    for i in range(n):\\n\"\n        \"        acc += i * 0.5\\n\"\n        \"    return acc\"\n    )\n    helper = \"def _sq(x: float) -> float:\\n    return x * x\\n\"\n    assert cythonize_function(helper) == \"cdef double _sq(double x):\\n    return x * x\"\n    for bad in (\"x = 1\", \"def a():\\n    pass\\ndef b():\\n    pass\\n\"):\n        try:\n            cythonize_function(bad)\n            assert False\n        except ValueError as e:\n            assert str(e) == \"one function expected\"")

## Warm-up

Python integers never overflow; a 32-bit C `int` does, silently wrapping round to a negative number. And Python's `//` rounds down (towards minus infinity) while C's integer division rounds towards zero. When Cython turns your Python into C for speed, these are the differences you have to keep in mind.

## What Cython is

You write a `.pyx` file in the Cython language, a superset of Python. The `cython` compiler translates it to a C file that uses the C API (Monday's reference counting, argument parsing and error handling, all generated correctly for you), and your C compiler turns that into an extension module you `import`.

```
# primes.pyx
def count_primes(int limit):          # the argument is a C int
    cdef int n, d, count = 0          # C locals: no Python objects
    cdef bint is_prime
    for n in range(2, limit):
        is_prime = True
        d = 2
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 1
        if is_prime:
            count += 1
    return count
```

Build it, on your own computer, with `pip install cython` and a `setup.py` containing `from Cython.Build import cythonize` and `setup(ext_modules=cythonize("primes.pyx"))`, then `pip install .`; or try it in a notebook with the `%%cython` cell magic. With the types, the loops become plain C and the function is typically tens of times faster than the Python version. Without any types, the same file still compiles, and is usually only modestly faster, because every operation still works on Python objects.

## Adding types

Cython has three kinds of function and a way to type variables:

- **`def`**: an ordinary Python function, callable from Python. Its typed arguments are converted on entry (`def f(int n)` raises `TypeError` or `OverflowError` for bad values, just as `PyArg_ParseTuple` would).
- **`cdef`**: a C function, callable only from Cython code, with no Python call overhead: `cdef double square(double x): return x * x`.
- **`cpdef`**: both: a fast C version for Cython callers and a Python wrapper for everyone else.
- **`cdef` variables**: `cdef int i`, `cdef double total = 0`, `cdef list items` (a typed Python object), `cdef int[:] view` (a typed memoryview).

C types to know: `int`, `long`, `long long`, `double`, `float`, `bint` (a C int used as a bool), `size_t`, `Py_ssize_t` (the right type for indices and lengths), and `char *`.

**Pure-Python mode** puts the same information in normal Python syntax, so the file stays valid Python and still runs (slowly) without compiling:

```
import cython

@cython.cfunc
def square(x: cython.double) -> cython.double:
    return x * x

def total(n: cython.int) -> cython.double:
    i: cython.int
    acc: cython.double = 0
    for i in range(n):
        acc += square(i)
    return acc
```

## What stays slow

Cython speeds up code by replacing Python operations with C ones. It can only do that where it **knows the types**. Anything it can't type stays a Python operation, as slow as in Python (or a little faster, since the interpreter loop is skipped). Typical causes:

- **Untyped variables in hot loops**: `total = 0` without `cdef` stays a Python int.
- **Calls to Python functions and methods** inside the loop (`items.append`, `math.sqrt` on objects, your own `def` functions): each is a full Python call. Use `cdef` functions and C library functions (`from libc.math cimport sqrt`).
- **Python containers**: indexing a `list` or `dict` works on objects. Use typed memoryviews or C arrays for numeric data.
- **Dynamic features**: `*args`, `**kwargs`, `getattr` with computed names, `eval`, closures, generators. They work, but at Python speed.

The tool for finding these is **`cython -a file.pyx`**, which writes an HTML **annotation report**: each line is shaded yellow according to how much it interacts with Python, and clicking a line shows the generated C. The goal is white lines inside the hot loops; yellow elsewhere doesn't matter. (Profile first, Week 65: only the hot loops need this attention.)

## Memoryviews and the buffer protocol

Numeric data in Python usually lives in a **buffer**: a block of raw memory inside a `bytes`, `bytearray`, `array.array`, NumPy array, or a C library's memory. The **buffer protocol** lets code see that memory directly, without copying, along with its element **format** (`"d"` for doubles, `"i"` for ints, `"B"` for unsigned bytes), item size and shape.

In Python, **`memoryview(obj)`** gives that direct view: slicing it doesn't copy, writing through it changes the original, and `.cast("i")` reinterprets the bytes as another format. In Cython, a **typed memoryview** such as `double[:] values` does the same at C speed: `values[i]` compiles to a plain memory access, which is why Cython code that takes NumPy arrays as `double[:]` arguments runs so fast.

```
def scale(double[:] values, double factor):
    cdef Py_ssize_t i
    for i in range(values.shape[0]):
        values[i] *= factor            # plain C: no Python objects
```

## C semantics and nogil

Typed Cython code follows **C's rules**, not Python's:

- **Fixed-size integers overflow** silently (the warm-up), so choose `long long` or check ranges; Cython's `overflowcheck` directive can catch it, at some cost.
- **Division**: by default Cython keeps Python's semantics for `//` and `%` on C integers (rounding down, and raising `ZeroDivisionError`), at a small cost. With the directive `cdivision=True`, it uses C's: rounding towards zero, a remainder with the sign of the dividend, and no check for zero (a crash instead).
- **Index checking**: memoryview indexing checks bounds and allows negative indices unless you turn that off with `boundscheck(False)` and `wraparound(False)`, which is faster and unsafe.

And because typed code needn't touch Python objects, it can **release the GIL** (Week 66): `with nogil:` around such a block lets other threads run Python meanwhile, and `prange` from `cython.parallel` runs a loop's iterations on several cores with OpenMP. That's how Cython code gets real parallelism inside one process.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, `pip install cython`, put `%load_ext cython` in a Jupyter notebook, and time the prime-counting function with and without its `cdef` lines in a `%%cython -a` cell; the report shows exactly which lines the types made white. Tomorrow: writing extensions in Rust with PyO3.